# Verify Symmetry Breaking

**Goal:** Determine mathematically why the Team Sum policy prefers Agent 1 to pick apples (70%) over Agent 0 (30%).

**Experiment:**
1. Construct a perfectly symmetric state (Both agents next to an apple).
2. Simulate Agent 0 picking -> Measure Resulting Value ($V_0 + V_1$).
3. Simulate Agent 1 picking -> Measure Resulting Value ($V_0 + V_1$).
4. Compare components to find the "Tilt".

In [1]:
# === CONFIGURATION ===
HEIGHT = 6
WIDTH = 6
NUM_AGENTS = 2
DISCOUNT = 0.99

CNN_CONV_CHANNELS = [32, 64]
CNN_HEAD_HIDDEN_DIM = 128
CNN_HEAD_NUM_LAYERS = 3
CNN_KERNEL_SIZE = 3

PATH_DECENTRALIZED_AG0 = "decen_cnn_-12_reward/model_agent0_step16050000.pt"
PATH_DECENTRALIZED_AG1 = "decen_cnn_-12_reward/model_agent1_step16050000.pt"

In [2]:
import torch
import numpy as np
import sys
sys.path.append("../") 
from models.value_cnn_new import ValueCNNDecentralizedStandard
from tadd_helpers.env_functions import init_empty_state, State
from orchard.environment import MoveAction

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

--- PyTorch is configured to use: cuda ---


In [3]:
# Load Models
dec_models = []
for p in [PATH_DECENTRALIZED_AG0, PATH_DECENTRALIZED_AG1]:
    m = ValueCNNDecentralizedStandard(HEIGHT, WIDTH, 0.0, DISCOUNT, CNN_HEAD_HIDDEN_DIM, CNN_HEAD_NUM_LAYERS, CNN_CONV_CHANNELS, CNN_KERNEL_SIZE).to(DEVICE)
    m.load_state_dict(torch.load(p, map_location=DEVICE))
    m.eval()
    dec_models.append(m)

In [4]:
def get_vals(state):
    v0 = dec_models[0].get_value(state, agent_pos=state.agent_position(0))
    v1 = dec_models[1].get_value(state, agent_pos=state.agent_position(1))
    return v0, v1, v0+v1


In [7]:
def analyze_full_action_distribution():
    print(f"\n{'='*60}")
    print("FULL ACTION SCAN: SYMMETRIC STATE")
    print(f"{'='*60}")
    
    # 1. Setup Symmetric State
    # Agents at top corners (0,0) and (0,5). Apples directly below them at (1,0) and (1,5).
    s_start = init_empty_state(HEIGHT, WIDTH, NUM_AGENTS)
    s_start.set_agent_position(0, np.array([0, 0]))
    s_start.set_agent_position(1, np.array([0, 5]))
    s_start.apples[:] = 0
    s_start.apples[1, 0] = 1
    s_start.apples[1, 5] = 1
    
    # 2. Define Actions (Down is PICK, Right/Left is AVOID, Stay is WAIT)
    actions = [
        ("DOWN (Pick)", MoveAction.DOWN),
        ("RIGHT (Move)", MoveAction.RIGHT),
        ("LEFT (Move)", MoveAction.LEFT),
        ("UP (Move)", MoveAction.UP),
        ("STAY (Wait)", MoveAction.STAY)
    ]
    
    # 3. Scan Both Agents
    for agent_id in [0, 1]:
        print(f"\n--- AGENT {agent_id} ACTION PREFERENCE ---")
        # Fixed formatting to avoid backslash error
        print(f"{'Action':<15} | {'Immediate R':<12} | {'V_Sum(Next)':<12} | {'Q_Team':<12} | {'V_0(Next)':<10} | {'V_1(Next)':<10}")
        print("-" * 85)
        
        best_q = -9999
        best_act_name = ""
        
        for name, move in actions:
            # Physics
            curr_pos = s_start.agent_position(agent_id)
            new_pos = np.clip(curr_pos + move.vector, [0, 0], [HEIGHT-1, WIDTH-1])
            
            s_next = s_start.copy()
            s_next.set_agent_position(agent_id, new_pos)
            
            # Reward Logic
            r = 0.0
            if s_start.apples[tuple(new_pos)] > 0: # Note: Check Old State's apples at New Pos
                r = 1.0
                s_next.remove_apple_at(new_pos)
                
            # Value Query
            v0, v1, v_sum = get_vals(s_next)
            
            # Q-Value
            q_val = r + DISCOUNT * v_sum
            
            # Log
            print(f"{name:<15} | {r:<12.1f} | {v_sum:<12.2f} | {q_val:<12.2f} | {v0:<10.2f} | {v1:<10.2f}")
            
            if q_val > best_q:
                best_q = q_val
                best_act_name = name
                
        print(f"\n>> WINNER for Agent {agent_id}: {best_act_name}")

analyze_full_action_distribution()


FULL ACTION SCAN: SYMMETRIC STATE

--- AGENT 0 ACTION PREFERENCE ---
Action          | Immediate R  | V_Sum(Next)  | Q_Team       | V_0(Next)  | V_1(Next) 
-------------------------------------------------------------------------------------
DOWN (Pick)     | 1.0          | 18.10        | 18.91        | 20.09      | -1.99     
RIGHT (Move)    | 0.0          | 17.77        | 17.59        | 19.95      | -2.19     
LEFT (Move)     | 0.0          | 17.81        | 17.63        | 19.90      | -2.09     
UP (Move)       | 0.0          | 17.81        | 17.63        | 19.90      | -2.09     
STAY (Wait)     | 0.0          | 17.81        | 17.63        | 19.90      | -2.09     

>> WINNER for Agent 0: DOWN (Pick)

--- AGENT 1 ACTION PREFERENCE ---
Action          | Immediate R  | V_Sum(Next)  | Q_Team       | V_0(Next)  | V_1(Next) 
-------------------------------------------------------------------------------------
DOWN (Pick)     | 1.0          | 16.72        | 17.55        | 18.69      | -1